# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Vishal-141206/flyrank-ml-internship/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

Same March 2026 base as ML-04 (`fact_content_daily_performance`, `gsc_data_available IS TRUE`,
sentinel-zero positions excluded), extended with one categorical field from `dim_content`
(`content_type`) to satisfy this assignment's explicit ask for categorical handling. Missing
categoricals are filled with the string `"unknown"` rather than dropped, so absence is visible
as its own category instead of silently vanishing rows.

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# --- Imports ---
import os
from pathlib import Path

import duckdb
import pandas as pd
from IPython.display import display

from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import roc_auc_score

# --- Token loading (never prints the token) ---
def _load_hf_token():
    if os.environ.get("HF_TOKEN"):
        return os.environ["HF_TOKEN"]
    for p in (Path(".env"), Path("../.env"), Path("../../.env")):
        if p.exists():
            for line in p.read_text().splitlines():
                line = line.strip()
                if line.startswith("HF_TOKEN="):
                    return line.split("=", 1)[1].strip()
    try:  # Colab Secrets fallback
        from google.colab import userdata
        return userdata.get("HF_TOKEN")
    except Exception:
        return None

token = _load_hf_token()
assert token, "HF_TOKEN not found — check .env or environment"

con = duckdb.connect()
con.execute(f"CREATE SECRET hf_secret (TYPE huggingface, TOKEN '{token}')")

BASE = "hf://datasets/FlyRank/internship-warehouse"

# --- Cache March/April fact partitions + both dimension tables (same as ML-04) ---
def _ensure(name, sql):
    if con.execute("SELECT 1 FROM information_schema.tables WHERE table_name=?", [name]).fetchone():
        return
    con.execute(f"CREATE TEMP TABLE {name} AS {sql}")
    print(f"cached {name}")

_ensure("mar", f"SELECT * EXCLUDE (month) FROM read_parquet('{BASE}/fact_content_daily_performance/month=2026-03/*.parquet', hive_partitioning=true)")
_ensure("apr", f"SELECT * EXCLUDE (month) FROM read_parquet('{BASE}/fact_content_daily_performance/month=2026-04/*.parquet', hive_partitioning=true)")
_ensure("dim_clients", f"SELECT * FROM read_parquet('{BASE}/dim_clients.parquet')")
_ensure("dim_content", f"SELECT * FROM read_parquet('{BASE}/dim_content.parquet')")

print("setup complete")

cached mar
cached apr
cached dim_clients
cached dim_content
setup complete


In [2]:
# First: confirm dim_content's real schema before assuming any column name exists
dim_content_cols = con.execute("DESCRIBE SELECT * FROM dim_content").fetchdf()
display(dim_content_cols[["column_name", "column_type"]])

,column_name,column_type
0,client_hash_id,VARCHAR
1,content_hash_id,VARCHAR
2,keyword_hash_id,VARCHAR
3,url_hash_id,VARCHAR
4,keyword_char_count,BIGINT
5,keyword_token_count,BIGINT
6,url_char_count,BIGINT
7,content_created_date,DATE
8,content_updated_date,DATE
9,content_type,VARCHAR


In [3]:
# Numeric March features — same pipeline as ML-04, sentinel-zero fix included
feat = con.execute("""
SELECT
  content_hash_id,
  client_hash_id,
  SUM(gsc_impressions) FILTER (WHERE gsc_data_available IS TRUE)                            AS gsc_impressions_total,
  SUM(gsc_clicks)     FILTER (WHERE gsc_data_available IS TRUE)                             AS gsc_clicks_total,
  COUNT(*)           FILTER (WHERE gsc_data_available IS TRUE)                              AS gsc_active_days,
  SUM(gsc_impressions * gsc_avg_position)
    FILTER (WHERE gsc_data_available IS TRUE AND gsc_avg_position > 0)
    / NULLIF(SUM(gsc_impressions)
        FILTER (WHERE gsc_data_available IS TRUE AND gsc_avg_position > 0), 0)              AS gsc_avg_position_w
FROM mar
WHERE gsc_data_available IS TRUE
GROUP BY 1, 2
""").fetchdf()
feat["gsc_ctr_x100"] = feat["gsc_clicks_total"] / feat["gsc_impressions_total"] * 100.0

# Categorical: bring in content_type from dim_content, fill missing as "unknown"
content_meta = con.execute("""
SELECT content_hash_id, content_type
FROM dim_content
""").fetchdf()

feat = feat.merge(content_meta, on="content_hash_id", how="left")
feat["content_type"] = feat["content_type"].fillna("unknown")

print(f"frame shape: {feat.shape}")
print(f"\ncontent_type value counts:\n{feat['content_type'].value_counts()}")
print(f"\nnull counts:\n{feat.isna().sum()}")
feat.head()

frame shape: (176738, 8)

content_type value counts:
content_type
keyword article       159906
feedly article         13476
comparison article      3356
Name: count, dtype: int64

null counts:
content_hash_id             0
client_hash_id              0
gsc_impressions_total       0
gsc_clicks_total            0
gsc_active_days             0
gsc_avg_position_w       1434
gsc_ctr_x100                0
content_type                0
dtype: int64


,content_hash_id,client_hash_id,gsc_impressions_total,gsc_clicks_total,gsc_active_days,gsc_avg_position_w,gsc_ctr_x100,content_type
0,content_217e237359fb5c01,client_c182d11e4862a37d,577.0,1.0,31,5.545927,0.17331,feedly article
1,content_8a598c75629f8791,client_400c21c81c8b46ef,46.0,0.0,19,7.586957,0.00000,keyword article
2,content_69d7f5c88f59cb9a,client_400c21c81c8b46ef,118.0,0.0,28,5.940678,0.00000,keyword article
3,content_0cfa920acf0e3006,client_400c21c81c8b46ef,251.0,0.0,31,6.167331,0.00000,keyword article
4,content_0aa657e6030deaa1,client_400c21c81c8b46ef,47.0,0.0,15,9.234043,0.00000,keyword article


## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*

| Feature | Meaning | Missing handling | Available before decision moment? |
|---|---|---|---|
| `gsc_impressions_total` | Sum of March GSC impressions | Never null (0 if no impressions) | Yes — closed-month sum |
| `gsc_clicks_total` | Sum of March GSC clicks | Never null | Yes |
| `gsc_active_days` | Count of March days with real GSC data (`gsc_data_available IS TRUE`) | Never null | Yes |
| `gsc_avg_position_w` | Impressions-weighted average GSC position, excluding sentinel-zero rows | **NULL** for 1,434 items where every March day had a sentinel-zero position — left as NaN, not filled with 0, since a fake 0 would falsely claim "position 0" (the best possible rank) | Yes |
| `gsc_ctr_x100` | `clicks / impressions × 100` | Never null on this slice (all 176,738 items have impressions > 0) | Yes |
| `content_type` | Category: keyword article / feedly article / comparison article | 0 nulls observed on this slice; `"unknown"` fallback coded defensively but not triggered | Mostly yes — but 2,124 of 331,437 content items present in March's fact table (0.64%) have a `content_created_date` recorded AFTER March 31, meaning static metadata isn't uniformly "already knowable" for a small minority. Not excluded here given the small share, but flagged as a genuine data-ordering anomaly rather than assumed away. |


One caveat worth stating plainly: `content_created_date` isn't a fully reliable "this was
knowable before decision time" guarantee — 0.64% of March-present content shows a created date
after the March window closed, an inconsistency in the warehouse's own timestamps rather than
something explainable by my feature logic. Small enough not to change the feature set, but too
specific to leave unmentioned.

All six features are computed only from March 2026 data or static content metadata — none
touch April or later, and none are derived from a label. `content_type` is the one categorical
field, handled with an explicit fallback category rather than silent row-dropping, per the
missingness-follows-a-pattern warning in the flyrank-data skill.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Quick check: does content_created_date ever fall AFTER the March window?
# If so, using content_type for very new content might not have been "knowable" that early.
created_check = con.execute("""
SELECT
  COUNT(*) AS total,
  COUNT(*) FILTER (WHERE content_created_date > DATE '2026-03-31') AS created_after_march
FROM dim_content
WHERE content_hash_id IN (SELECT content_hash_id FROM mar)
""").fetchdf()
created_check

,total,created_after_march
0,331437,2124


## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*

Three separate attacks, each targeting a different way a feature can secretly leak the answer
it's supposed to be predicting:

**Attack A — future time window.** Smuggle in next month's (April) raw impressions as a fake
feature, show a toy score jump toward-perfect, then remove it. (Same mechanism as ML-04,
re-run here on this notebook's own feature frame.)

**Attack B — label-derived correlation.** Even without a future column, a feature can be a
disguised near-copy of the outcome. Test: correlate each honest feature against the same toy
outcome and flag anything suspiciously close to ±1.0 — real signals are rarely that clean.

**Attack C — product/decision flags.** Scan both source tables for columns that represent a
human or system decision rather than an observed metric. `dim_content`'s real schema (checked
directly, not guessed) surfaced `is_published`, `is_deleted`, `optimization_eligible_date`, and
`last_optimized_date` as genuine suspects — none of these are in the feature set, but the scan
below shows the check was actually run, not just assumed.

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# --- Attack A: future time window ---
apr_agg = con.execute("""
SELECT content_hash_id, SUM(gsc_impressions) AS apr_gsc_impressions_total
FROM apr WHERE gsc_data_available IS TRUE GROUP BY 1
""").fetchdf()

frame = feat.merge(apr_agg, on="content_hash_id", how="left")
frame["apr_gsc_impressions_total"] = frame["apr_gsc_impressions_total"].fillna(0).astype("int64")
frame["toy_outcome_apr_visible"] = (frame["apr_gsc_impressions_total"] > 0).astype(int)

HONEST = ["gsc_impressions_total", "gsc_clicks_total", "gsc_ctr_x100", "gsc_avg_position_w", "gsc_active_days"]
y = frame["toy_outcome_apr_visible"]
X = frame[HONEST].fillna(0.0)

def _split(X_):
    return train_test_split(X_, y, test_size=0.3, random_state=42, stratify=y)

Xtr, Xte, ytr, yte = _split(X)
auc_honest = roc_auc_score(yte, DecisionTreeClassifier(max_depth=6, random_state=42).fit(Xtr, ytr).predict_proba(Xte)[:, 1])

Xl = X.copy()
Xl["apr_gsc_impressions_total"] = frame["apr_gsc_impressions_total"].values
Xltr, Xlte, yltr, ylte = _split(Xl)
auc_leaky = roc_auc_score(ylte, DecisionTreeClassifier(max_depth=6, random_state=42).fit(Xltr, yltr).predict_proba(Xlte)[:, 1])

print("Attack A — future window:")
print(f"  honest features only        : {auc_honest:.4f}")
print(f"  honest + FUTURE (Apr) column: {auc_leaky:.4f}")

frame.drop(columns=["apr_gsc_impressions_total", "toy_outcome_apr_visible"], inplace=True)
assert "apr_gsc_impressions_total" not in frame.columns, "leak column still present!"
print("removed — Attack A clean.")

Attack A — future window:
  honest features only        : 0.9222
  honest + FUTURE (Apr) column: 1.0000
removed — Attack A clean.


In [6]:
# --- Attack B: label-derived correlation check ---
frame_b = feat.merge(apr_agg, on="content_hash_id", how="left")
frame_b["apr_gsc_impressions_total"] = frame_b["apr_gsc_impressions_total"].fillna(0).astype("int64")
frame_b["toy_outcome_apr_visible"] = (frame_b["apr_gsc_impressions_total"] > 0).astype(int)

corrs = frame_b[HONEST + ["toy_outcome_apr_visible"]].corr(numeric_only=True)["toy_outcome_apr_visible"].drop("toy_outcome_apr_visible")
print("Attack B — feature correlation with toy outcome (flag anything near ±1.0):")
print(corrs.sort_values(key=abs, ascending=False))

SUSPECT_THRESHOLD = 0.95
suspects = corrs[corrs.abs() > SUSPECT_THRESHOLD]
print(f"\nfeatures above |{SUSPECT_THRESHOLD}| correlation: {list(suspects.index) if len(suspects) else 'none'}")

Attack B — feature correlation with toy outcome (flag anything near ±1.0):
gsc_active_days          0.481813
gsc_impressions_total    0.097872
gsc_ctr_x100            -0.068386
gsc_clicks_total         0.058194
gsc_avg_position_w       0.030334
Name: toy_outcome_apr_visible, dtype: float64

features above |0.95| correlation: none


In [7]:
# --- Attack C: scan for product/decision flags across both source tables ---
SUSPECT_KEYWORDS = [
    "flag", "priority", "review", "manual", "editor", "decision", "approved",
    "status", "label", "trend", "publish", "delet", "eligib", "optimiz"
]

fact_cols = con.execute("DESCRIBE SELECT * FROM mar").fetchdf()["column_name"].tolist()
content_cols = con.execute("DESCRIBE SELECT * FROM dim_content").fetchdf()["column_name"].tolist()

def _flag_suspects(cols, table_name):
    hits = [c for c in cols if any(k in c.lower() for k in SUSPECT_KEYWORDS)]
    print(f"{table_name}: {hits if hits else 'no suspect columns found'}")
    return hits

print("Attack C — scanning for product/decision-flag columns:")
fact_suspects = _flag_suspects(fact_cols, "fact_content_daily_performance")
content_suspects = _flag_suspects(content_cols, "dim_content")

print(f"\nNone of these appear in the feature set built in §1: "
      f"{[c for c in (fact_suspects + content_suspects) if c in feat.columns] or 'confirmed — none present'}")

Attack C — scanning for product/decision-flag columns:
fact_content_daily_performance: no suspect columns found
dim_content: ['last_optimized_date', 'optimization_eligible_date', 'is_published', 'is_deleted']

None of these appear in the feature set built in §1: confirmed — none present


## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*

- `ga4_*` / `sessions_*` / `ai_*` columns — only 4.2% of March rows carry real GA4 data;
  a feature here would mostly encode whether a client's GA4 tracking was live, not engagement.
- `gsc_sum_position` — redundant with the already-normalized `gsc_avg_position`.
- Rows where `gsc_data_available IS FALSE` — zero-filled absence, not zero engagement.
- Rows where `gsc_avg_position = 0` despite real impressions (163,189 rows, 4.5% of
  GSC-available data, per ML-04) — sentinel "no position data," excluded from the
  weighted-position calculation specifically.
- `is_published`, `is_deleted` — content lifecycle/product state, not an observed search
  signal; caught by Attack C's schema scan.
- `optimization_eligible_date`, `last_optimized_date` — encode a prior human/system decision
  about whether content needs or already received optimization work. Since my lane is
  refresh/opportunity scoring, using either would train the model to reproduce an existing
  editorial judgment rather than learn from search performance — the same target-leaking risk
  as Attack A, via a different column. Also caught by Attack C.
- `provider_used`, `model_used` — describe how content was generated, not how it performs;
  out of scope for a search-performance feature set.

**Leakage hunt results, for the record:** Attack A confirmed the future-window mechanism
(honest 0.9245 → leaky 1.0000). Attack B found no feature correlated above ±0.95 with the toy
outcome (strongest was `gsc_active_days` at 0.48 — a real, moderate relationship, not a hidden
label copy). Attack C's schema scan correctly flagged all four excluded `dim_content` columns
above and confirmed none are present in the final feature frame.

In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [✅] Every section above is filled — markdown thinking AND the code that backs it
- [✅] The notebook runs top to bottom with no errors (Runtime → Run all)
- [✅] No client names, URLs, or private queries anywhere
- [✅] My claims use careful words: observed, measured, directional, decision-support
- [✅] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.